# Face Indexing with VideoDB

Detect, cluster, and identify faces in any video — then find where each person appears.

**Pipeline:**
1. **Understand** — detect faces in every frame
2. **Index** — cluster detections into identities (people)
3. **Browse** — view identities, name them, find their appearances

In [ ]:
!pip install git+https://github.com/video-db/videodb-python.git@indexing-v2 \
    pillow requests

In [ ]:
import videodb

conn = videodb.connect(
    base_url="https://api.videodb.io",
    api_key="YOUR_API_KEY",
)
coll = conn.get_collection()

In [ ]:
video = coll.upload("https://www.youtube.com/watch?v=vVlEVRKv4is")
video.play()

In [ ]:
print(video.id)
# Or load an existing video:
# video = coll.get_video("m-z-YOUR-VIDEO-ID")

## Step 1: Face Understanding (Detection)

`video.understand()` extracts frames from the video and runs face detection on each frame.
It returns an `understanding_id` immediately — use `get_understanding()` to poll until detection is complete.

In [ ]:
# Create Understanding Job
understanding_id = video.understand(extract=["faces"], store=True)
print("Understanding ID:", understanding_id)

In [ ]:
# Fetch Understanding Job (Polling)
understanding = video.get_understanding(understanding_id)
print(understanding)

In [ ]:
total_faces = sum(len(seg.detections) for seg in understanding.results)
segments_with_faces = sum(1 for s in understanding.results if s.detections)

print(f"Total faces detected: {total_faces}")
print(f"Segments with faces: {segments_with_faces}/{len(understanding.results)}")

for seg in understanding.results:
    if seg.detections:
        print(f"\nSample — {seg.timestamp_ms / 1000:.1f}s: {len(seg.detections)} face(s)")
        for det in seg.detections:
            print(f"  bbox={det.bbox}, confidence={det.confidence}")
        break

## Step 2: Face Indexing (Clustering)

`video.index()` takes the detected faces from understanding and clusters them into **identities** (people).
Faces that look similar are grouped together. Each identity gets a representative face and an embedding in the vector store.

In [ ]:
# Create Indexing Job
index_id = video.index(
    source={
        "type": "understanding",
        "understanding_id": understanding.id,
    },
)

print("Index ID:", index_id)

In [ ]:
# Fetch Indexing Job (Polling)
index = video.get_index(index_id)
print(index)

## Step 3: Browse the Face Store

After indexing, the **Face Store** contains:
- **Identities** — each represents a person, sorted by face count (most frequent first)
- **Faces** — individual detections linked to an identity, with timestamp, bounding box, and confidence

In [ ]:
import requests
from io import BytesIO

from IPython.display import display
from PIL import Image


def get_face_crop(coll, face_id, size=80):
    """Download frame, crop the face bounding box, and return a thumbnail."""
    face = coll.face_store.faces.get(face_id)
    resp = requests.get(face.frame_url, timeout=30)
    img = Image.open(BytesIO(resp.content)).convert("RGB")

    x, y, w, h = [float(v) for v in face.bbox]
    crop = img.crop((x, y, x + w, y + h))
    crop.thumbnail((size, size))
    return crop


def show_faces_row(crops, gap=8, pad=4):
    """Display a horizontal row of face crops."""
    if not crops:
        return

    cell_w = max(c.width for c in crops) + pad * 2
    cell_h = max(c.height for c in crops) + pad * 2
    total_w = cell_w * len(crops) + gap * (len(crops) - 1)
    row = Image.new("RGB", (total_w, cell_h), (30, 30, 30))

    x_off = 0
    for c in crops:
        cx = x_off + (cell_w - c.width) // 2
        cy = (cell_h - c.height) // 2
        row.paste(c, (cx, cy))
        x_off += cell_w + gap

    display(row)

In [ ]:
identities = coll.face_store.identities.list()
print(f"Found {len(identities)} identities\n")

for identity in identities[:10]:
    crops = [
        get_face_crop(coll, fid) for fid in identity.representative_faces
    ]
    show_faces_row(crops)
    print(
        f"  ID: {identity.id}  |  Name: {identity.name}"
        f"  |  Faces: {identity.face_count}"
    )
    print()

In [ ]:
top_identity = identities[0]
faces = coll.face_store.faces.list(identity_id=top_identity.id)

label = top_identity.name or top_identity.id
print(f"Identity '{label}' has {len(faces)} faces:\n")

for face in faces[:5]:
    print(
        f"  Face {face.id}  |  ts={face.timestamp_ms}ms"
        f"  |  confidence={face.confidence}"
    )

## Step 4: Name Identities

After reviewing the face crops above, label each identity with a name.
This makes it easy to refer to them later.

In [ ]:
# Name the top identities after visual review
identities[0].update(name="Person A")
identities[1].update(name="Person B")

# Verify
for identity in identities[:2]:
    print(f"  {identity.id} → {identity.name}")

## Step 5: Find Where a Person Appears

Use the face store to list all face detections for a specific identity in this video.
Each face record has a `timestamp_ms` telling you exactly when that person was on screen.

In [ ]:
targets = identities[:2]

faces_by_person = {}
for target in targets:
    faces_by_person[target.id] = coll.face_store.faces.list(
        identity_id=target.id,
        video_id=video.id,
    )
    label = target.name or target.id
    print(f'"{label}" — {len(faces_by_person[target.id])} detections')
    for face in faces_by_person[target.id][:5]:
        print(f"  {face.timestamp_ms / 1000:.1f}s  (confidence={face.confidence})")
    print()

### Generate a Compilation Stream


#### For Person A


Use `video.generate_stream()` with a timeline of `(start, end)` tuples to create a playable stream of just the segments where the person appears.

In [ ]:
def build_timeline(faces, segment_duration=1.0):
    """Convert face detections to merged (start, end) segments."""
    timestamps = sorted(set(f.timestamp_ms / 1000.0 for f in faces))
    raw = [(ts, ts + segment_duration) for ts in timestamps]
    merged = []
    for start, end in raw:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return raw, merged


for target in targets:
    faces = faces_by_person[target.id]
    label = target.name or target.id
    raw, timeline = build_timeline(faces)

    print(f'"{label}": {len(raw)} detections → {len(timeline)} merged segments')
    for start, end in timeline[:5]:
        print(f"  {start:.1f}s - {end:.1f}s")

    stream_url = video.generate_stream(timeline=timeline)
    print(f"  Stream: {stream_url}")
    print(f"  Player: https://console.videodb.io/player?url={stream_url}")
    print()

## Advanced Configuration

The steps above used all defaults. Both `understand()` and `index()` accept config dicts to tune detection and clustering.

**Understanding config** (`config.faces`):

| Parameter | Default | Description |
|---|---|---|
| `confidence_threshold` | 0.75 | Minimum YOLO detection confidence (lower = more faces, more false positives) |
| `min_face_size` | 40 | Minimum face width/height in pixels (lower = detect smaller/distant faces) |

**Index config** (`config.identity`):

| Parameter | Default | Description |
|---|---|---|
| `match_threshold` | 0.82 | Cosine similarity threshold for clustering faces into the same identity (lower = more aggressive merging) |

**Segmentation / Sampling** (passed to `understand()`):

| Parameter | Default | Description |
|---|---|---|
| `segmentation.window` | `"1s"` | Time window per segment |
| `sampling.frame_count` | 2 | Frames extracted per segment |


In [ ]:
understanding_id_v2 = video.understand(
    extract=["faces"],
    config={
        "faces": {
            "confidence_threshold": 0.6,
            "min_face_size": 30,
        }
    },
    segmentation={"type": "time", "window": "1s"},
    sampling={"frame_count": 2},
    store=True,
)

print("Understanding ID:", understanding_id_v2)

In [ ]:
# Fetch Understanding Job (Polling)
understanding = video.get_understanding(understanding_id_v2)
print(understanding)

In [ ]:
index_id_v2 = video.index(
    source={
        "type": "understanding",
        "understanding_id": understanding_id_v2,
    },
    config={
        "identity": {
            "match_threshold": 0.70,
        },
    },
    name="my-face-index-v2",
)

print("Index ID:", index_id_v2)


In [ ]:
# Fetch Indexing Job (Polling)
index = video.get_index(index_id_v2)
print(index)

## Utilities

In [ ]:
# List all understandings for this video
understandings = video.list_understanding(extract=["faces"])
print("Understandings:", understandings)

# List all indexes for this video
indexes = video.list_indexes()
print("Indexes:", indexes)


In [ ]:
# Update the representative face for an identity
# (pick a face_id from the faces list above)
# identity.update(set_representative={"face_id": "FACE_ID_HERE"})

In [ ]:
# Get a specific face by ID
face = coll.face_store.faces.get(face_id=faces[0].id)
print(face)
print(f"  frame_url: {face.frame_url}")
print(f"  bbox: {face.bbox}")
print(f"  timestamp_ms: {face.timestamp_ms}")